# We will use two key abstractions from LangChain

1. LLM - We can use LangChain to kind of wrap an LLM 

2. Retriever - Abstraction around Chroma, the data store and the embedding model, all sort of wrapped up into this retriever

~ Many common abstractions like this can be called with invoke()

## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [2]:
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings

import gradio as gr

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")

print("Groq key loaded:", groq_api_key[:8] + "...")

Groq key loaded: gsk_TdT2...


In [4]:
MODEL = "llama-3.3-70b-versatile"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [6]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [7]:
retriever.invoke("Who is Avery?")

[Document(id='3da9d7fc-17b9-4e3b-9284-869fed2c6878', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [8]:
response = llm.invoke("Who is Avery?")
print(response.content)

I don't have enough information to determine who Avery is. Could you please provide more context or details about Avery? This will help me better understand your question and provide a more accurate response.


## Time to put this together!

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [10]:
def answer_question(question, history):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        context=context
    )

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])

    return response.content

In [11]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has been instrumental in guiding it to its current position as a leading Insurance Tech provider. Before launching Insurellm, Avery worked as a Senior Product Manager at Innovate Insurance Solutions, where she developed innovative insurance products for the tech sector.'

## What could possibly come next? 😂

In [12]:
gr.ChatInterface(
    fn=answer_question,
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!

## Implementation directory has two modules -

~ ingest.py

1. Read in Knowledge Base

2. Turn documents into chunks

3. Vectorize the chunks

4. Store in Chroma

~ answer.py

1. fetch_context(question): for a given question, we'll get relevant context

2. answer_question(question, history): for a given question and some history, it will fetch the relevant context and answer the question

### How to use this:

1. Gradio application app.py

2. uv run ingest.py

3. uv run app.py